# Starter — a figure that derives its own title

*Replace this notebook. It exists to show the house rules working, not to be kept.*

The rule this demonstrates is the one in this folder's README:

> Every published figure should be derivable, not retyped. A number typed into a
> document is a copy, and copies drift silently.

So the headline number in the chart title is **computed from the same frame the
chart is drawn from**, and a cell at the end fails if the two ever disagree.


In [ ]:
import os

import pandas as pd
from sqlalchemy import create_engine

# The PG* variables are already set by the dev container. Nothing here
# holds a host or a password, so this notebook is the same on every machine.
engine = create_engine(
    'postgresql+psycopg://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}'.format(**os.environ)
)


## The question

Acme Council's premise is that critical knowledge is undocumented. That is a
claim about two columns, so it can be checked rather than asserted.


In [ ]:
assets = pd.read_sql(
    '''select ka.title, ka.asset_type, ka.criticality,
              ka.documentation_coverage as coverage
         from knowledge_asset ka
        order by ka.criticality desc, coverage''',
    engine,
)
assets


In [ ]:
# The message, derived. Nothing below retypes it.
most_critical = assets[assets.criticality >= 4]
gap = round(100 - most_critical.coverage.mean())
message = f'The most critical assets are {gap} points short of full documentation'
message


## The chart

IBCS conventions, as the README asks: a message-first title, one accent colour
against grey, units in the axis label, and a source note under the exhibit.


In [ ]:
import matplotlib.pyplot as plt

ACCENT, GREY = '#1F4E79', '#C9CFD6'
colours = [ACCENT if c >= 4 else GREY for c in assets.criticality]

fig, ax = plt.subplots(figsize=(8, 3.2), dpi=140)
ax.barh(assets.title, assets.coverage, color=colours)
ax.set_title(message, loc='left', fontsize=11, fontweight='bold')
ax.set_xlabel('Documentation coverage, %')
ax.set_xlim(0, 100)
ax.invert_yaxis()
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
fig.text(0.01, -0.02, 'Source: acme_sad.knowledge_asset — synthetic placeholder data',
         fontsize=7, color='#5B6770')
fig.tight_layout()


## The check

This is what makes the figure derivable rather than merely correct today. If
someone edits the title by hand, or the data moves under it, this fails.


In [ ]:
recomputed = round(100 - assets[assets.criticality >= 4].coverage.mean())
assert f'{recomputed} points' in message, (
    f'the title says something the data does not: {message!r} vs {recomputed}'
)
print('title agrees with the data:', message)
